# Election Bloc Change Prediction Project
## Notebook 04 — Historical socioeconomic feature engineering

Loads historical and recent demographic, income, welfare and education workbooks. Features are aligned as-of the latest source year **strictly before** the current election year, preventing use of information published after an election.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time, re, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 300)
REPO_URL = "https://github.com/IfatDav/Election_Bloc_Prediction_Project.git"
DEFAULT_REPO = Path("/content/Election_Bloc_Prediction_Project")

def locate_repo():
    candidates=[]
    if os.getenv("ELECTION_PROJECT_ROOT"):
        candidates.append(Path(os.environ["ELECTION_PROJECT_ROOT"]).expanduser())
    cwd=Path.cwd().resolve()
    candidates += [cwd, *cwd.parents, DEFAULT_REPO]
    for p in candidates:
        if (p/"data"/"raw").exists(): return p
    if Path("/content").exists():
        if DEFAULT_REPO.exists(): shutil.rmtree(DEFAULT_REPO)
        subprocess.run(["git","clone","--depth","1",REPO_URL,str(DEFAULT_REPO)],check=True)
        return DEFAULT_REPO
    raise FileNotFoundError("Repository not found. Set ELECTION_PROJECT_ROOT.")

REPO_ROOT=locate_repo()
RAW_DIR=REPO_ROOT/"data"/"raw"
INTERIM_DIR=REPO_ROOT/"data"/"interim"
PROCESSED_DIR=REPO_ROOT/"data"/"processed"
REPORTS_DIR=REPO_ROOT/"reports"
TABLES_DIR=REPORTS_DIR/"tables"
FIGURES_DIR=REPORTS_DIR/"figures"
SUMMARIES_DIR=REPORTS_DIR/"summaries"
MODELS_DIR=REPO_ROOT/"models"
for d in [INTERIM_DIR,PROCESSED_DIR,TABLES_DIR,FIGURES_DIR,SUMMARIES_DIR,MODELS_DIR]: d.mkdir(parents=True,exist_ok=True)
print("Repository:",REPO_ROOT)

In [ ]:
PANEL=INTERIM_DIR/"election_transition_panel_with_type.csv"
if not PANEL.exists() and (REPO_ROOT/".git").exists(): subprocess.run(["git","-C",str(REPO_ROOT),"pull","--ff-only","origin","main"],check=True)
panel=pd.read_csv(PANEL,dtype={"locality_symbol":"string"},low_memory=False)
SOURCE_CONFIG={
 "demographic":(RAW_DIR/"Demographic_data","Demographic_{year}.xlsx",[2003,2006,2009,2013,2015,2019,2020,2021,2022,2023]),
 "income":(RAW_DIR/"Average_income","ICBS_{year}.xlsx",[2003,2006,2009,2013,2015,2019,2020,2021,2022,2023]),
 "welfare":(RAW_DIR/"Unemployment_data","Unemployment_{year}.xlsx",[2003,2006,2009,2013,2015,2019,2020,2021,2022,2023]),
 "education":(RAW_DIR/"Education_data","Education_{year}.xlsx",[2016,2019,2020,2021,2022,2023]),
}

## Robust workbook extraction and schema normalization

In [ ]:
MISSING={"","..","...","-","--","—","nan","none"}
def clean_col(x):
    x=str(x).replace("\ufeff","").replace("\n"," ").strip();x=re.sub(r"\s+"," ",x);return x
def canonical_feature_name(x,year):
    x=clean_col(x);x=re.sub(rf"\b{year}\b","",x);x=re.sub(r"תש[א-ת]\"?[א-ת]?","",x);x=re.sub(r"\s+"," ",x).strip(" -_")
    x=x.replace("גילאי","בני").replace("4-0","0-4").replace("9-5","5-9").replace("14-10","10-14").replace("19-15","15-19")
    return x
def read_best(path):
    xls=pd.ExcelFile(path);best=None;score=-1
    for sheet in xls.sheet_names:
        for header in range(0,6):
            try: df=pd.read_excel(path,sheet_name=sheet,header=header)
            except Exception: continue
            df.columns=[clean_col(c) for c in df.columns]
            if not any("סמל" in c and "יישוב" in c for c in df.columns): continue
            numeric=sum(pd.to_numeric(df[c],errors="coerce").notna().sum()>5 for c in df.columns)
            s=len(df)*max(numeric,1)
            if s>score: best=(df,sheet,header);score=s
    if best is None: raise ValueError(f"No locality table in {path}")
    return best

def extract_source(path,source,year):
    df,sheet,header=read_best(path)
    sc=next(c for c in df.columns if "סמל" in c and "יישוב" in c);nc=next((c for c in df.columns if "שם" in c and "יישוב" in c),None)
    out=pd.DataFrame({"locality_symbol":pd.to_numeric(df[sc],errors="coerce").astype("Int64").astype("string"),"source_year":year})
    if nc: out["source_locality_name"]=df[nc].astype("string")
    used=set(out.columns)
    for c in df.columns:
        if c in {sc,nc}: continue
        name=f"{source}__{canonical_feature_name(c,year)}"
        if name in used: name=f"{name}__dup"
        used.add(name)
        s=df[c].astype("string").str.strip().replace(list(MISSING),pd.NA)
        values=pd.to_numeric(s.str.replace(",","",regex=False).str.replace("%","",regex=False),errors="coerce")
        if values.notna().sum()>=5: out[name]=values.astype(float)
    out=out.loc[out.locality_symbol.notna()&out.locality_symbol.ne("<NA>")].drop_duplicates("locality_symbol")
    return out,{"source":source,"year":year,"file":path.name,"sheet":sheet,"header_row":header,"rows":len(out),"numeric_features":out.shape[1]-3}

source_tables={}; audits=[]
for source,(folder,pattern,years) in SOURCE_CONFIG.items():
    parts=[]
    for year in years:
        path=folder/pattern.format(year=year)
        if not path.exists(): continue
        part,audit=extract_source(path,source,year);parts.append(part);audits.append(audit)
    if not parts: raise FileNotFoundError(f"No files found for {source}")
    source_tables[source]=pd.concat(parts,ignore_index=True,sort=False)
pd.DataFrame(audits)

## As-of alignment to each transition

In [ ]:
def latest_before(years,election_year):
    eligible=[y for y in years if y<election_year]
    return max(eligible) if eligible else None
model=panel.copy(); coverage=[]
for source,table in source_tables.items():
    years=sorted(table.source_year.unique())
    feature_cols=[c for c in table.columns if c not in {"locality_symbol","source_year","source_locality_name"}]
    model[f"previous_{source}_feature_year"]=model.previous_year.map(lambda y:latest_before(years,int(y)))
    model[f"current_{source}_feature_year"]=model.current_year.map(lambda y:latest_before(years,int(y)))
    prev=table.rename(columns={c:f"previous_feature__{c}" for c in feature_cols}).rename(columns={"source_year":f"previous_{source}_feature_year"}).drop(columns=["source_locality_name"],errors="ignore")
    curr=table.rename(columns={c:f"current_feature__{c}" for c in feature_cols}).rename(columns={"source_year":f"current_{source}_feature_year"}).drop(columns=["source_locality_name"],errors="ignore")
    model=model.merge(prev,on=["locality_symbol",f"previous_{source}_feature_year"],how="left").merge(curr,on=["locality_symbol",f"current_{source}_feature_year"],how="left")
    for c in feature_cols:
        p=f"previous_feature__{c}";q=f"current_feature__{c}";model[f"delta_feature__{c}"]=model[q]-model[p]
    current_present=model[[f"current_feature__{c}" for c in feature_cols]].notna().any(axis=1)
    previous_present=model[[f"previous_feature__{c}" for c in feature_cols]].notna().any(axis=1)
    model[f"has_current_{source}"]=current_present;model[f"has_previous_{source}"]=previous_present
    for t,g in model.groupby("transition_id"): coverage.append({"source":source,"transition_id":t,"rows":len(g),"current_rows":int(g[f"has_current_{source}"].sum()),"previous_rows":int(g[f"has_previous_{source}"].sum())})
core=["demographic","income","welfare"]
model["has_current_core_features"]=model[[f"has_current_{s}" for s in core]].all(axis=1)
model["has_complete_annual_pair"]=model[[f"has_current_{s}" for s in core]+[f"has_previous_{s}" for s in core]].all(axis=1)
model["previous_log_valid_votes"]=np.log1p(pd.to_numeric(model.previous_valid_votes,errors="coerce"))
print(model.groupby("transition_id")[["has_current_core_features","has_complete_annual_pair"]].mean().mul(100).round(1))

## Feature manifest and outputs

In [ ]:
MODELED_BLOCS=["Right","Center_Left","Haredi","Arab"]
history=[*[f"previous_clr_{b}" for b in MODELED_BLOCS],"previous_turnout_pct","previous_log_valid_votes"]
previous=[c for c in model.columns if c.startswith("previous_feature__")];current=[c for c in model.columns if c.startswith("current_feature__")];delta=[c for c in model.columns if c.startswith("delta_feature__")]
manifest={"target_columns":[f"delta_clr_{b}" for b in MODELED_BLOCS],"history_feature_columns":history,"static_feature_columns":["locality_type","analysis_group"],"previous_annual_feature_columns":previous,"current_annual_feature_columns":current,"delta_annual_feature_columns":delta,"feature_sets":{"history_only":history+["locality_type"],"history_plus_levels":history+["locality_type"]+current,"history_plus_levels_and_change":history+["locality_type"]+current+delta},"final_test_transition":"K24_to_K25","alignment_rule":"latest source year strictly before election year"}
paths={"annual":INTERIM_DIR/"locality_annual_features_historical.csv","model":INTERIM_DIR/"modeling_transition_features.csv","availability":TABLES_DIR/"historical_feature_coverage.csv","schema":TABLES_DIR/"historical_source_schema_audit.csv","manifest":SUMMARIES_DIR/"feature_manifest.json","summary":SUMMARIES_DIR/"notebook_04_summary.json"}
# Save source tables in one sparse long file.
annual=pd.concat([t.assign(source=s) for s,t in source_tables.items()],ignore_index=True,sort=False)
annual.to_csv(paths["annual"],index=False,encoding="utf-8-sig");model.to_csv(paths["model"],index=False,encoding="utf-8-sig");pd.DataFrame(coverage).to_csv(paths["availability"],index=False,encoding="utf-8-sig");pd.DataFrame(audits).to_csv(paths["schema"],index=False,encoding="utf-8-sig");paths["manifest"].write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")
summary={"notebook":"04_feature_engineering","rows":len(model),"transitions":sorted(model.transition_id.unique()),"current_core_eligible_rows":int(model.has_current_core_features.sum()),"complete_pair_rows":int(model.has_complete_annual_pair.sum()),"feature_counts":{"history":len(history),"current":len(current),"delta":len(delta)},"final_test_status":"K24_to_K25 features prepared but target remains locked"}
paths["summary"].write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(summary,ensure_ascii=False,indent=2))